In [3]:
import pandas as pd
import os

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# dynamicProperties

Checking dynamic properties

In [6]:
# Load the parquet file
mof_df = pd.read_parquet("All_data/merged_all_filtered.parquet")

# Get unique dynamicProperties entries (drop NaN) and save to CSV
unique_dynamicProperties = (
    mof_df["dynamicProperties"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

unique_dynamicProperties.to_csv("All_data/dynamicProperties_unique.csv", index=False, header=["dynamicProperties"])

print(f"Saved {len(unique_dynamicProperties)} unique dynamicProperties entries to All_data/dynamicProperties_unique.csv")

Saved 8035 unique dynamicProperties entries to All_data/dynamicProperties_unique.csv


## individualCount

In [7]:
# check correspondence between dynamicProperties containing "observedindividualcount=" and individualCount
merged = pd.read_parquet("All_data/merged_all_filtered.parquet")
dp = merged["dynamicProperties"].astype(str)
ic = merged["individualCount"]

dp_mask = dp.str.contains("observedindividualcount=", case=False, na=False)
ic_mask = ic.notna() & (ic != -999)

print("Rows with 'observedindividualcount=' in dynamicProperties:", int(dp_mask.sum()))
print("Rows with individualCount present (not NA and not -999):", int(ic_mask.sum()))

print("\nContingency table (dp_has_obs_indiv x individualCount_present):")
print(pd.crosstab(dp_mask, ic_mask))

# examples where dp has observedindividualcount but individualCount is missing
print("\nExamples where dynamicProperties contains observedindividualcount but individualCount is missing:")
print(merged.loc[dp_mask & ~ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# examples where individualCount present but dp does NOT contain observedindividualcount
print("\nExamples where individualCount present but dynamicProperties does NOT contain observedindividualcount:")
print(merged.loc[~dp_mask & ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# extract numeric observedindividualcount from dynamicProperties and compare when both present
extracted = dp.str.extract(r"(?i)observedindividualcount=([0-9]+(?:\.[0-9]+)?)")[0].astype(float)
both_mask = dp_mask & ic_mask & extracted.notna()
print(f"\nRows where both present and extracted value parsed: {int(both_mask.sum())}")

if both_mask.any():
    diff = (merged.loc[both_mask, "individualCount"].astype(float) - extracted[both_mask]).abs()
    equal_count = int((diff == 0).sum())
    diff_count = int((diff != 0).sum())
    print(f" - exact matches between individualCount and extracted observedindividualcount: {equal_count}")
    print(f" - mismatches: {diff_count}")
    if diff_count:
        print("\nExamples of mismatches:")
        print(
            pd.DataFrame({
                "dynamicProperties": merged.loc[both_mask, "dynamicProperties"].str.slice(0,200),
                "individualCount": merged.loc[both_mask, "individualCount"],
                "extracted_observedindividualcount": extracted[both_mask]
            }).head(10).to_string(index=False)
        )

Rows with 'observedindividualcount=' in dynamicProperties: 23432
Rows with individualCount present (not NA and not -999): 2986321

Contingency table (dp_has_obs_indiv x individualCount_present):
individualCount     False    True 
dynamicProperties                 
False              951157  2977069
True                14180     9252

Examples where dynamicProperties contains observedindividualcount but individualCount is missing:
                                                                dynamicProperties  individualCount
                  observedweight=0.03;temperature=9.22;observedindividualcount=8;              NaN
observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;              NaN
                                                     observedindividualcount=147;              NaN
    observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;              NaN
                                                     observedindividual

In [11]:
# check correspondence between dynamicProperties containing "observedindividualcount=" and individualCount
dp = merged["dynamicProperties"].astype(str)
ic = merged["individualCount"]

dp_mask = dp.str.contains("SubSampleNumbers=", case=False, na=False)
ic_mask = ic.notna() & (ic != -999)

print("Rows with 'SubSampleNumbers=' in dynamicProperties:", int(dp_mask.sum()))
print("Rows with individualCount present (not NA and not -999):", int(ic_mask.sum()))

print("\nContingency table (dp_has_obs_indiv x individualCount_present):")
print(pd.crosstab(dp_mask, ic_mask))

# examples where dp has SubSampleNumbers but individualCount is missing
print("\nExamples where dynamicProperties contains SubSampleNumbers but individualCount is missing:")
print(merged.loc[dp_mask & ~ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# examples where individualCount present but dp does NOT contain SubSampleNumbers
print("\nExamples where individualCount present but dynamicProperties does NOT contain SubSampleNumbers:")
print(merged.loc[~dp_mask & ic_mask, ["dynamicProperties", "individualCount"]].head(10).to_string(index=False))

# extract numeric SubSampleNumbers from dynamicProperties and compare when both present
extracted = dp.str.extract(r"(?i)SubSampleNumbers=([0-9]+(?:\.[0-9]+)?)")[0].astype(float)
both_mask = dp_mask & ic_mask & extracted.notna()
print(f"\nRows where both present and extracted value parsed: {int(both_mask.sum())}")

if both_mask.any():
    diff = (merged.loc[both_mask, "individualCount"].astype(float) - extracted[both_mask]).abs()
    equal_count = int((diff == 0).sum())
    diff_count = int((diff != 0).sum())
    print(f" - exact matches between individualCount and extracted SubSampleNumbers: {equal_count}")
    print(f" - mismatches: {diff_count}")
    if diff_count:
        print("\nExamples of mismatches:")
        print(
            pd.DataFrame({
                "dynamicProperties": merged.loc[both_mask, "dynamicProperties"].str.slice(0,200),
                "individualCount": merged.loc[both_mask, "individualCount"],
                "extracted_SubSampleNumbers": extracted[both_mask]
            }).head(10).to_string(index=False)
        )

Rows with 'SubSampleNumbers=' in dynamicProperties: 867
Rows with individualCount present (not NA and not -999): 2986321

Contingency table (dp_has_obs_indiv x individualCount_present):
individualCount     False    True 
dynamicProperties                 
False              965224  2985567
True                  113      754

Examples where dynamicProperties contains SubSampleNumbers but individualCount is missing:
                                                                                                                                                                                        dynamicProperties  individualCount
                 SubSampleWeightInGrams=0.002; SubSampleNumbers=; CalculatedWeightInGrams=0.002; CalculatedNumbers=; SampledAll=Y; AverageWeightInGrams=0.0005; NonSubSubWeightInGrams=; NonSubSubNumber=              NaN
          SubSampleWeightInGrams=0.041999999; SubSampleNumbers=; CalculatedWeightInGrams=0.041999999; CalculatedNumbers=; SampledAll=Y; AverageW

In [8]:
# Fill individualCount from dynamicProperties when individualCount is NaN
merged_all_mof = merged.copy()

observed_ic = (
    merged_all_mof["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)observedindividualcount=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = merged_all_mof["individualCount"].isna() & observed_ic.notna()

merged_all_mof.loc[fill_mask, "individualCount"] = observed_ic[fill_mask]

print(f"Filled {int(fill_mask.sum())} individualCount values from observedindividualcount.")

preview_cols = ["dynamicProperties", "individualCount"]
print(merged_all_mof.loc[fill_mask, preview_cols].head(10).to_string(index=False))

merged_all_mof.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 14180 individualCount values from observedindividualcount.
                                                                dynamicProperties  individualCount
                  observedweight=0.03;temperature=9.22;observedindividualcount=8;              8.0
observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;              3.5
                                                     observedindividualcount=147;            147.0
    observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;             12.0
                                                     observedindividualcount=147;            147.0
                  observedweight=0.07;temperature=9.22;observedindividualcount=4;              4.0
                                                       observedindividualcount=4;              4.0
                                                       observedindividualcount=3;              3.0
                 observedweight=0.07;temper

In [10]:
# Fill individualCount from dynamicProperties when individualCount is NaN
merged_all_mof = merged.copy()

observed_ic = (
    merged_all_mof["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)SubSampleNumbers=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = merged_all_mof["individualCount"].isna() & observed_ic.notna()

merged_all_mof.loc[fill_mask, "individualCount"] = observed_ic[fill_mask]

print(f"Filled {int(fill_mask.sum())} individualCount values from SubSampleNumbers.")

preview_cols = ["dynamicProperties", "individualCount"]
print(merged_all_mof.loc[fill_mask, preview_cols].head(10).to_string(index=False))

merged_all_mof.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 0 individualCount values from SubSampleNumbers.
Empty DataFrame
Columns: [dynamicProperties, individualCount]
Index: []
Saved merged_all_mof.parquet


## biomass

In [9]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)observedweight=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]

print(f"Filled {int(fill_mask.sum())} biomass values from observedweight.")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 4896 biomass values from observedweight.
                                                                 dynamicProperties  biomass
                   observedweight=0.03;temperature=9.22;observedindividualcount=8;     0.03
 observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;     0.01
     observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;     0.00
                   observedweight=0.07;temperature=9.22;observedindividualcount=4;     0.07
                  observedweight=0.07;temperature=6.9;observedindividualcount=182;     0.07
     observedweight=0;samplesize=0.26;temperature=-1.57;observedindividualcount=4;     0.00
 observedweight=0.01;samplesize=0.26;temperature=-1.58;observedindividualcount=36;     0.01
observedweight=0.04;samplesize=0.26;temperature=-1.44;observedindividualcount=616;     0.04
 observedweight=0.02;samplesize=0.39;temperature=-1.1;observedindividualcount=2.5;     0.02
 observedweight=0.02;samplesize=

In [12]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)SubSampleWeightInGrams=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]
mof_df.loc[fill_mask, "biomassUnit"] = "g"

print(f"Filled {int(fill_mask.sum())} biomass values from SubSampleWeightInGrams.")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 867 biomass values from SubSampleWeightInGrams.
                                                                                                                                                                                            dynamicProperties  biomass
   SubSampleWeightInGrams=0.048999999; SubSampleNumbers=51; CalculatedWeightInGrams=0.282999992; CalculatedNumbers=300; SampledAll=N; AverageWeightInGrams=0.00096; NonSubSubWeightInGrams=; NonSubSubNumber=    0.049
                    SubSampleWeightInGrams=0.001; SubSampleNumbers=1; CalculatedWeightInGrams=0.001; CalculatedNumbers=1; SampledAll=N; AverageWeightInGrams=0.001; NonSubSubWeightInGrams=; NonSubSubNumber=    0.001
SubSampleWeightInGrams=0.037999999; SubSampleNumbers=11; CalculatedWeightInGrams=0.037999999; CalculatedNumbers=11; SampledAll=N; AverageWeightInGrams=0.003454545; NonSubSubWeightInGrams=; NonSubSubNumber=    0.038
                   SubSampleWeightInGrams=0.014; SubSampleNumbers=4; CalculatedWeight

In [13]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)ObservedWeight=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]

print(f"Filled {int(fill_mask.sum())} biomass values from ObservedWeight.")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 4896 biomass values from ObservedWeight.
                                                                 dynamicProperties  biomass
                   observedweight=0.03;temperature=9.22;observedindividualcount=8;     0.03
 observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;     0.01
     observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;     0.00
                   observedweight=0.07;temperature=9.22;observedindividualcount=4;     0.07
                  observedweight=0.07;temperature=6.9;observedindividualcount=182;     0.07
     observedweight=0;samplesize=0.26;temperature=-1.57;observedindividualcount=4;     0.00
 observedweight=0.01;samplesize=0.26;temperature=-1.58;observedindividualcount=36;     0.01
observedweight=0.04;samplesize=0.26;temperature=-1.44;observedindividualcount=616;     0.04
 observedweight=0.02;samplesize=0.39;temperature=-1.1;observedindividualcount=2.5;     0.02
 observedweight=0.02;samplesize=

In [14]:
# Fill biomass from observedweight in dynamicProperties when biomass is empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

biomass_empty = mof_df["biomass"].isna() | (mof_df["biomass"].astype("string").str.strip() == "")
observed_weight = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r"(?i)Mass (kg)=([0-9]+(?:\.[0-9]+)?)")[0]
    .astype(float)
)

fill_mask = biomass_empty & observed_weight.notna()
mof_df.loc[fill_mask, "biomass"] = observed_weight[fill_mask]
mof_df.loc[fill_mask, "biomassUnit"] = "kg"

print(f"Filled {int(fill_mask.sum())} biomass values from Mass (kg).")

preview_cols = ["dynamicProperties", "biomass"]
print(mof_df.loc[fill_mask, preview_cols].head(10).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 0 biomass values from Mass (kg).
Empty DataFrame
Columns: [dynamicProperties, biomass]
Index: []
Saved merged_all_mof.parquet


## areaSampled

In [18]:
# Fill areaSampled and areaSampledUnit from SampleSize in dynamicProperties when empty

mof_df = pd.read_parquet("merged_all_mof.parquet")

area_sampled_empty = mof_df["areaSampled"].isna() | (mof_df["areaSampled"].astype("string").str.strip() == "")
area_sampled_unit_empty = mof_df["areaSampledUnit"].isna() | (mof_df["areaSampledUnit"].astype("string").str.strip() == "")

sample_size_raw = (
    mof_df["dynamicProperties"]
    .astype("string")
    .str.extract(r'(?i)SampleSize=([^;\"]+)')[0]
    .str.strip()
    .str.strip('"')
)

sample_size_parts = sample_size_raw.str.extract(r'^\s*([0-9]+(?:[.,][0-9]+)?)\s*(.*)$')
sample_size_value = sample_size_parts[0].str.replace(',', '.', regex=False).astype(float)
sample_size_unit = sample_size_parts[1].fillna(sample_size_raw).str.strip()

fill_area_mask = area_sampled_empty & sample_size_value.notna()
fill_unit_mask = area_sampled_unit_empty & sample_size_unit.notna()

mof_df.loc[fill_area_mask, "areaSampled"] = sample_size_value[fill_area_mask]
mof_df.loc[fill_unit_mask, "areaSampledUnit"] = sample_size_unit[fill_unit_mask]

print(f"Filled {int(fill_area_mask.sum())} areaSampled values and {int(fill_unit_mask.sum())} areaSampledUnit values from SampleSize.")

preview_cols = ["dynamicProperties", "areaSampled", "areaSampledUnit"]
print(mof_df.loc[fill_area_mask | fill_unit_mask, preview_cols].head(30).to_string(index=False))

mof_df.to_parquet("merged_all_mof.parquet", index=False)
print("Saved merged_all_mof.parquet")

Filled 0 areaSampled values and 4173 areaSampledUnit values from SampleSize.
                                                                 dynamicProperties  areaSampled areaSampledUnit
 observedweight=0.01;samplesize=0.3;temperature=10.02;observedindividualcount=3.5;         0.30                
     observedweight=0;samplesize=0.25;temperature=-0.8;observedindividualcount=12;         0.25                
                                                                  samplesize=0.75;         0.75                
                                                                   samplesize=2.3;         2.30                
                                                                  samplesize=4.97;         4.97                
     observedweight=0;samplesize=0.26;temperature=-1.57;observedindividualcount=4;         0.26                
 observedweight=0.01;samplesize=0.26;temperature=-1.58;observedindividualcount=36;         0.26                
observedweight=0.04;samples

In [19]:
unique_area_sampled_units = (
    mof_df["areaSampledUnit"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(f"Unique areaSampledUnit values: {len(unique_area_sampled_units)}")
print(unique_area_sampled_units.to_string(index=False))

Unique areaSampledUnit values: 11
              
             #
           #/l
        #/m**2
             %
 Not available
           cm²
ind per 1000m2
 ind per 300m2
            m2
            m²


In [20]:
mask = mof_df["areaSampledUnit"].astype(str).str.strip() == "Not available"
mof_df.loc[mask, "areaSampledUnit"] = None
print(f"Replaced {int(mask.sum())} 'Not available' entries in areaSampledUnit with NaN")
mof_df.to_parquet("merged_all_mof.parquet", index=False)

Replaced 67 'Not available' entries in areaSampledUnit with NaN


# OBIS MoF

In [ ]:
obis = pd.read_parquet('OBIS/mof_OBIS.parquet')
merged = pd.read_parquet('All_data/merged_all_mof.parquet')

obis_filtered = obis[obis['id'].isin(merged['sourceID'])].copy()

removed_count = len(obis) - len(obis_filtered)
print(f"Removed {removed_count} records")

obis_filtered.to_parquet('OBIS/mof_OBIS_filtered.parquet', index=False)

Removed 4164642 records


In [ ]:
# Load the parquet file
mof_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")

# Get unique measurementType + measurementUnit pairs and save to CSV
unique_measurement_pairs = (
    mof_df[["measurementType", "measurementUnit"]]
    .dropna(subset=["measurementType"])
    .drop_duplicates()
    .sort_values(["measurementType", "measurementUnit"], na_position="last")
    .reset_index(drop=True)
)

unique_measurement_pairs.to_csv(
    "OBIS/measurementType_unique.csv", index=False, header=["measurementType", "measurementUnit"]
)

print(
    f"Saved {len(unique_measurement_pairs)} unique measurementType/measurementUnit pairs to measurementType_unique.csv"
)

Saved 337 unique measurementType/measurementUnit pairs to measurementType_unique.csv


## individualCount

In [22]:
# Fill individualCount in All_data/merged_all_mof.parquet from OBIS measurementValue
# when measurementType indicates abundance/count and individualCount is empty.

target_types = {
    "abundnr",
    "individualcount",
    "count",
    "count (in assayed sample) of biological entity specified elsewhere",
    "observedindividualcount",
    "taxon abundance",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("merged_all_mof.parquet")

# Keep only relevant measurementType rows (case-insensitive)
obis_counts = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric
obis_counts["measurementValue_num"] = pd.to_numeric(
    obis_counts["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# One value per id (first non-null numeric value)
id_to_count = (
    obis_counts.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Treat empty as NaN or -999
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)

# Map OBIS id -> merged sourceID
mapped_counts = merged_df["sourceID"].astype("string").map(id_to_count)

fill_mask = empty_ic_mask & mapped_counts.notna()
merged_df.loc[fill_mask, "individualCount"] = mapped_counts[fill_mask]

added_count = int(fill_mask.sum())
print(f"Added {added_count} individualCount values from OBIS measurementValue.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 13847 individualCount values from OBIS measurementValue.
Saved All_data/merged_all_mof.parquet


In [23]:
# Fill individualCount (and areaSampledUnit) in All_data/merged_all_mof.parquet
# from OBIS/mof_OBIS_filtered.parquet when measurementType indicates abundance.

target_types = {
    "abundance",
    "abundance of biological entity specified elsewhere per unit area of the bed",
    "abundance of biological entity specified elsewhere per unit volume of the water body",
    "abundance_[#/m3]",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter abundance-like measurement types (case-insensitive)
obis_abund = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric for individualCount
obis_abund["measurementValue_num"] = pd.to_numeric(
    obis_abund["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_abund.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_count = id_map["measurementValue_num"]
id_to_unit = id_map["measurementUnit"].astype("string")

# Empty masks
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)
empty_asu_mask = merged_df["areaSampledUnit"].isna() | (
    merged_df["areaSampledUnit"].astype("string").str.strip() == ""
)

# Map OBIS id -> merged sourceID
mapped_count = merged_df["sourceID"].astype("string").map(id_to_count)
mapped_unit = merged_df["sourceID"].astype("string").map(id_to_unit)

# Fill
fill_ic_mask = empty_ic_mask & mapped_count.notna()
fill_asu_mask = empty_asu_mask & mapped_unit.notna()

merged_df.loc[fill_ic_mask, "individualCount"] = mapped_count[fill_ic_mask]
merged_df.loc[fill_asu_mask, "areaSampledUnit"] = mapped_unit[fill_asu_mask]

print(f"Added {int(fill_ic_mask.sum())} individualCount values from OBIS abundance measurementValue.")
print(f"Added {int(fill_asu_mask.sum())} areaSampledUnit values from OBIS abundance measurementUnit.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 79070 individualCount values from OBIS abundance measurementValue.
Added 79271 areaSampledUnit values from OBIS abundance measurementUnit.
Saved All_data/merged_all_mof.parquet


In [24]:
# Fill individualCount, areaSampled, and areaSampledUnit from count_per_1000m2 in OBIS

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for count_per_1000m2 measurement type (case-insensitive)
obis_count_1000m2 = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower() == "count_per_1000m2"
].copy()

# Parse measurementValue as numeric
obis_count_1000m2["measurementValue_num"] = pd.to_numeric(
    obis_count_1000m2["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_count_1000m2.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_count = id_map["measurementValue_num"]

# Empty masks
empty_ic_mask = merged_df["individualCount"].isna() | (merged_df["individualCount"] == -999)
empty_area_mask = merged_df["areaSampled"].isna() | (merged_df["areaSampled"].astype("string").str.strip() == "")
empty_unit_mask = merged_df["areaSampledUnit"].isna() | (merged_df["areaSampledUnit"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_count = merged_df["sourceID"].astype("string").map(id_to_count)

# Fill
fill_ic_mask = empty_ic_mask & mapped_count.notna()
fill_area_mask = empty_area_mask & mapped_count.notna()
fill_unit_mask = empty_unit_mask & mapped_count.notna()

merged_df.loc[fill_ic_mask, "individualCount"] = mapped_count[fill_ic_mask]
merged_df.loc[fill_area_mask, "areaSampled"] = 1000.0
merged_df.loc[fill_unit_mask, "areaSampledUnit"] = "m2"

added_count = int(fill_ic_mask.sum())
print(f"Added {added_count} entries from count_per_1000m2")
print(f" - {int(fill_ic_mask.sum())} individualCount values")
print(f" - {int(fill_area_mask.sum())} areaSampled values (set to 1000)")
print(f" - {int(fill_unit_mask.sum())} areaSampledUnit values (set to m2)")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 2331 entries from count_per_1000m2
 - 2331 individualCount values
 - 2331 areaSampled values (set to 1000)
 - 2331 areaSampledUnit values (set to m2)
Saved All_data/merged_all_mof.parquet


## biomass

In [25]:
# Fill biomass and biomassUnit from OBIS measurementValue when biomass is empty

target_types = {
    "biomass",
    "mass",
    "total macrofaunal biomass",
    "wet weight biomass (in assayed sample) of biological entity specified elsewhere",
    "wet weight biomass of biological entity specified elsewhere per unit area of the bed",
    "weight_flesh",
    "weight_whole",
    "wet weight",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter biomass-like measurement types (case-insensitive)
obis_biomass = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric
obis_biomass["measurementValue_num"] = pd.to_numeric(
    obis_biomass["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_biomass.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_biomass = id_map["measurementValue_num"]
id_to_unit = id_map["measurementUnit"].astype("string")

# Empty masks
empty_biomass_mask = merged_df["biomass"].isna() | (merged_df["biomass"].astype("string").str.strip() == "")
empty_biomass_unit_mask = merged_df["biomassUnit"].isna() | (merged_df["biomassUnit"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_biomass = merged_df["sourceID"].astype("string").map(id_to_biomass)
mapped_unit = merged_df["sourceID"].astype("string").map(id_to_unit)

# Fill
fill_biomass_mask = empty_biomass_mask & mapped_biomass.notna()
fill_unit_mask = empty_biomass_unit_mask & mapped_unit.notna()

merged_df.loc[fill_biomass_mask, "biomass"] = mapped_biomass[fill_biomass_mask]
merged_df.loc[fill_unit_mask, "biomassUnit"] = mapped_unit[fill_unit_mask]

print(f"Added {int(fill_biomass_mask.sum())} biomass values from OBIS measurementValue.")
print(f"Added {int(fill_unit_mask.sum())} biomassUnit values from OBIS measurementUnit.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 109248 biomass values from OBIS measurementValue.
Added 109248 biomassUnit values from OBIS measurementUnit.
Saved All_data/merged_all_mof.parquet


In [37]:
# Fill biomass and biomassUnit from OBIS measurementValue for specific measurement types

target_types = {
    "weight_g_per_1000m2",
    "biomass_[mg dw/m3]",
    "polychaete biomass",
}

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for target measurement types (case-insensitive)
obis_weight = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

# Parse measurementValue as numeric
obis_weight["measurementValue_num"] = pd.to_numeric(
    obis_weight["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_map = (
    obis_weight.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")
)

id_to_biomass = id_map["measurementValue_num"]
id_to_type = id_map["measurementType"].astype("string").str.lower()

# Empty biomass mask
empty_biomass_mask = merged_df["biomass"].isna() | (merged_df["biomass"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_biomass = merged_df["sourceID"].astype("string").map(id_to_biomass)
mapped_type = merged_df["sourceID"].astype("string").map(id_to_type)

# Fill biomass
fill_biomass_mask = empty_biomass_mask & mapped_biomass.notna()
merged_df.loc[fill_biomass_mask, "biomass"] = mapped_biomass[fill_biomass_mask]

# Fill biomassUnit based on type
weight_g_mask = fill_biomass_mask & (mapped_type == "weight_g_per_1000m2")
biomass_mg_mask = fill_biomass_mask & (mapped_type == "biomass_[mg dw/m3]")
polychaete_mask = fill_biomass_mask & (mapped_type == "polychaete biomass")

merged_df.loc[weight_g_mask, "biomassUnit"] = "g/1000 m2"
merged_df.loc[biomass_mg_mask, "biomassUnit"] = "mg dw/m3"
merged_df.loc[polychaete_mask, "biomassUnit"] = "polychaete biomass (g)"

added_count = int(fill_biomass_mask.sum())
print(f"Added {added_count} biomass values from OBIS.")
print(f" - {int(weight_g_mask.sum())} entries from weight_g_per_1000m2 (unit: g/1000 m2)")
print(f" - {int(biomass_mg_mask.sum())} entries from Biomass_[mg dw/m3] (unit: mg dw/m3)")
print(f" - {int(polychaete_mask.sum())} entries from Polychaete biomass (unit: polychaete biomass (g))")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 0 biomass values from OBIS.
 - 0 entries from weight_g_per_1000m2 (unit: g/1000 m2)
 - 0 entries from Biomass_[mg dw/m3] (unit: mg dw/m3)
 - 0 entries from Polychaete biomass (unit: polychaete biomass (g))
Saved All_data/merged_all_mof.parquet


## End coordinates

In [41]:
# Fill decimalLatitudeStop and decimalLongitudeStop from OBIS mof data

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for EndLatitude and EndLongitude
obis_end_lat = obis_df[obis_df["measurementType"] == "EndLatitude"].copy()
obis_end_lon = obis_df[obis_df["measurementType"] == "EndLongitude"].copy()

# Parse measurementValue as numeric
obis_end_lat["measurementValue_num"] = pd.to_numeric(
    obis_end_lat["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)
obis_end_lon["measurementValue_num"] = pd.to_numeric(
    obis_end_lon["measurementValue"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce",
)

# Keep one row per id (first valid numeric value)
id_to_lat = (
    obis_end_lat.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

id_to_lon = (
    obis_end_lon.dropna(subset=["measurementValue_num"])
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_num"]
)

# Empty masks
empty_lat_mask = merged_df["decimalLatitudeStop"].isna()
empty_lon_mask = merged_df["decimalLongitudeStop"].isna()

# Map OBIS id -> merged sourceID
mapped_lat = merged_df["sourceID"].astype("string").map(id_to_lat)
mapped_lon = merged_df["sourceID"].astype("string").map(id_to_lon)

# Fill
fill_lat_mask = empty_lat_mask & mapped_lat.notna()
fill_lon_mask = empty_lon_mask & mapped_lon.notna()

merged_df.loc[fill_lat_mask, "decimalLatitudeStop"] = mapped_lat[fill_lat_mask]
merged_df.loc[fill_lon_mask, "decimalLongitudeStop"] = mapped_lon[fill_lon_mask]

print(f"Added {int(fill_lat_mask.sum())} decimalLatitudeStop values from OBIS EndLatitude.")
print(f"Added {int(fill_lon_mask.sum())} decimalLongitudeStop values from OBIS EndLongitude.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 8913 decimalLatitudeStop values from OBIS EndLatitude.
Added 8913 decimalLongitudeStop values from OBIS EndLongitude.
Saved All_data/merged_all_mof.parquet


## Gear

In [42]:
# Fill gear in All_data/merged_all_mof.parquet from OBIS measurementValue for gear-like measurementType
obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

targets = {
    "equipment",
    "gear",
    "gear_type",
    "sampling instrument",
    "sampling instrument name",
    "gear name",
}

mt = obis_df["measurementType"].astype("string").str.strip().str.lower()
gear_rows = obis_df[mt.isin(targets)].copy()
gear_rows["measurementValue_str"] = gear_rows["measurementValue"].astype("string").str.strip()

# one value per id (first non-null, non-empty)
id_to_gear = (
    gear_rows[gear_rows["measurementValue_str"].notna() & (gear_rows["measurementValue_str"] != "")]
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_str"]
)

empty_gear_mask = merged_df["gear"].isna() | (merged_df["gear"].astype("string").str.strip() == "")
mapped_gear = merged_df["sourceID"].astype("string").map(id_to_gear)

fill_mask = empty_gear_mask & mapped_gear.notna()
merged_df.loc[fill_mask, "gear"] = mapped_gear[fill_mask]

added = int(fill_mask.sum())
print(f"Added {added} gear entries from OBIS measurementValue.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 9878 gear entries from OBIS measurementValue.
Saved All_data/merged_all_mof.parquet


## Sex

In [43]:
# Fill sex in All_data/merged_all_mof.parquet from OBIS measurementValue when sex is empty

obis_df = pd.read_parquet("OBIS/mof_OBIS_filtered.parquet")
merged_df = pd.read_parquet("All_data/merged_all_mof.parquet")

# Filter for sex-related measurement types (case-insensitive)
target_types = {
    "gender of biological entity specified elsewhere",
    "sex",
}

sex_rows = obis_df[
    obis_df["measurementType"].astype("string").str.strip().str.lower().isin(target_types)
].copy()

sex_rows["measurementValue_str"] = sex_rows["measurementValue"].astype("string").str.strip()

# One value per id (first non-null, non-empty)
id_to_sex = (
    sex_rows[sex_rows["measurementValue_str"].notna() & (sex_rows["measurementValue_str"] != "")]
    .drop_duplicates(subset=["id"], keep="first")
    .set_index("id")["measurementValue_str"]
)

# Empty sex mask
empty_sex_mask = merged_df["sex"].isna() | (merged_df["sex"].astype("string").str.strip() == "")

# Map OBIS id -> merged sourceID
mapped_sex = merged_df["sourceID"].astype("string").map(id_to_sex)

# Fill
fill_mask = empty_sex_mask & mapped_sex.notna()
merged_df.loc[fill_mask, "sex"] = mapped_sex[fill_mask]

added = int(fill_mask.sum())
print(f"Added {added} sex entries from OBIS measurementValue.")

merged_df.to_parquet("All_data/merged_all_mof.parquet", index=False)
print("Saved All_data/merged_all_mof.parquet")

Added 529 sex entries from OBIS measurementValue.
Saved All_data/merged_all_mof.parquet
